In [ ]:
import os
import pandas as pd
import glob
import re
import gzip
import ast
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm import tqdm
from copy import deepcopy
from itertools import combinations
from collections import defaultdict
from pymsaviz import MsaViz
from Bio import AlignIO, Align, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

In [39]:
#functions
def get_snp_positions(row, snps_set_col):
    enh_start = int(row['start'])
    
    snps = ast.literal_eval(row[snps_set_col])
    snps_set = {int(i) for i in snps}
    snp_positions = []
    
    for s in snps_set:
        snp_pos = s - enh_start
        snp_positions.append(snp_pos)

    return snp_positions

def read_fasta(path):
    with open(path) as fasta:
        fasta = fasta.read().strip().split('>')[1:]
        fasta = [f.split('\n', 1) for f in fasta]
        fasta = {f[0]: f[1].replace('\n','').strip() for f in fasta}
    return fasta

def seq_makeup(pathway):
    basename = os.path.split(pathway)
    out_dir = basename[0]
    base = basename[1].split('.')[0]
    
    file = read_fasta(pathway)
    
    right_order = ['hg38'] + species_dict['Primates'] + [
        i for i in sum(species_dict.values(), []) if i not in species_dict['Primates']]
    
    reordered_file = {k: file[k].upper() for k in right_order if k in file}

    back_to_fasta = []
    for k, v in reordered_file.items():
        v = str(v)
        if set(v) == {'-'}:
            continue
        sequence = Seq(v)
        back_to_fasta.append(SeqRecord(sequence, id=str(k), description=""))
        
    with open(f'{fasta_for_plots}/{base}.fasta', 'w') as handle:
        SeqIO.write(back_to_fasta, handle, 'fasta')

    return f'{fasta_for_plots}/{base}.fasta'


from itertools import combinations

def split_sites_for_msaviz(df, tf_col="motif_name", start_col="start", stop_col="stop",
                           len_col="motif_length", min_recip_overlap=0.1):
    work = df.reset_index(drop=True)
    adj = {i: set() for i in work.index}

    for i, j in combinations(work.index, 2):
        ov = max(0, min(work.loc[i, stop_col], work.loc[j, stop_col]) -
                    max(work.loc[i, start_col], work.loc[j, start_col]) + 1)
        if ov / work.loc[i, len_col] >= min_recip_overlap and ov / work.loc[j, len_col] >= min_recip_overlap:
            adj[i].add(j)
            adj[j].add(i)

    seen, groups = set(), []
    for i in work.index:
        if i in seen:
            continue
        stack, comp = [i], []
        seen.add(i)
        while stack:
            x = stack.pop()
            comp.append(x)
            for y in adj[x]:
                if y not in seen:
                    seen.add(y)
                    stack.append(y)
        groups.append(comp)

    singletons, merged = [], []
    for g in groups:
        sub = work.loc[g]
        item = (sub[start_col].min(), sub[stop_col].max(), ", ".join(dict.fromkeys(sub[tf_col])))
        (singletons if len(g) == 1 else merged).append(item)

    return singletons, merged

def split_sites_for_msaviz_df(df, tf_col="motif_name", start_col="start", stop_col="stop",
                              len_col="motif_length", min_recip_overlap=0.9):
    s, m = split_sites_for_msaviz(df, tf_col, start_col, stop_col, len_col, min_recip_overlap)
    import pandas as pd
    return (pd.DataFrame(s, columns=["start", "stop", "label"]),
            pd.DataFrame(m, columns=["start", "stop", "label"]))

In [27]:
# input
adata = ad.read_h5ad('../data/Enhancers_AnnData.h5ad')
df = adata.var.loc[adata.var['category'].isin(['primate_specific', 'human_unique']), :]

file_names = df.index.to_list() #for opening .fasta files with alignments
fasta_path='../output/1_2_FASTA_per_enh'
motif_path = '../output/2_2H_Motifs_per_enhancer'

In [ ]:
# output directores
plot_msa_output_path = '../plots/H_MSA_per_enhancer'
fasta_for_plots = '../output/MSA_FASTA_for_plots'

os.makedirs(fasta_for_plots, exist_ok=True)
os.makedirs(plot_msa_output_path, exist_ok=True)

In [ ]:
# getting species names
adata.obs.to_dict()['order']
species_dict = defaultdict(list)

for k, v in adata.obs.to_dict()['order'].items():
    species_dict[v].append(k)

In [ ]:
# MSA visualisations
for i, row in tqdm(df.iterrows()):
    enh = i
    motifs = pd.read_csv(f"{motif_path}/{enh}.csv", sep=";", header=0)
    alignment = f"{fasta_path}/{enh}.fa"

    msa = seq_makeup(alignment)
    
    mv = MsaViz(msa, wrap_length=100, show_grid=True,
                color_scheme="Nucleotide", show_consensus=True)

    # split motif sites into singletons and merged overlaps
    singletons, merged = split_sites_for_msaviz(
        motifs,
        tf_col="motif_name",
        start_col="start",
        stop_col="stop",
        len_col="motif_length",
        min_recip_overlap=0.1
    )

    # add singleton annotations
    for start, stop, label in singletons:
        mv.add_text_annotation((start, stop), label)

    # add merged annotations
    for start, stop, label in merged:
        mv.add_text_annotation((start, stop), label)

    # adding SNPs
    if pd.notna(row["snps"]):
        mv.add_markers(get_snp_positions(row, "all_snps"), color="black")
    
    mv.savefig(f"{plot_msa_output_path}/{enh}.png")

89it [32:45, 22.08s/it]
